# Qwen3-8B Answer Generation

# Qwen3-8B Answer Generation

This notebook generates answers to student questions using the `Qwen/Qwen3-8B` model and the previously saved top-10 results from the TF-IDF, BM25, and dense retrieval methods. For each question, the first `CONTEXT_K` ranked chunks are included in the RAG prompt.

Qwen3-8B is used in a zero-shot setting without additional fine-tuning. During development, answers are generated only for the validation set. The test set is processed after the prompt and generation parameters have been finalized.

The notebook is intended to run in Google Colab and requires:

- a CUDA GPU runtime;
- the `transformers`, `accelerate`, and `bitsandbytes` libraries;
- access to the project files stored on Google Drive;
- previously generated retrieval results in `data/retrieval/`.

The model is loaded in 4-bit quantized form to reduce GPU memory usage. Generated answers are saved incrementally so that the process can continue after an interruption.

## Libraries and Configuration


In [10]:
%pip install -q -U --no-cache-dir transformers accelerate "bitsandbytes>=0.46.1"

In [11]:
import json
from pathlib import Path
import sys
import bitsandbytes as bnb

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

In [12]:
print("Python:", sys.version)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Nema GPU-a")
print("bitsandbytes:", bnb.__version__)

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
CUDA: True
GPU: Tesla T4
bitsandbytes: 0.50.2


In [13]:
GENERATOR_NAME = "qwen"
GENERATOR_MODEL_NAME = "Qwen/Qwen3-8B"
GENERATOR_RUNTIME = "transformers_colab_4bit"

RETRIEVERS = ("tfidf", "bm25", "dense")
TOP_K_AVAILABLE = 10
CONTEXT_K = 5
MAX_NEW_TOKENS = 300

RUN_TEST = False
SPLITS = ("validation", "test") if RUN_TEST else ("validation",)

## Loading TF-IDF, BM25, and Dense Retrieval Results

In [14]:
from google.colab import drive

drive.mount("/content/drive")

# Change this path if the project folder has a different name or location.
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Student-Question-Answering-from-Course-Materials"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_ROOT}"
    )
RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"
GENERATION_DIR = PROJECT_ROOT / "data" / "generation" / "qwen_colab"


def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Neispravan JSON u redu {line_number}: {path}"
                ) from error

    return records


def validate_retrieval_records(records: list[dict], path: Path) -> None:
    required_record_fields = {
        "question_id", "question", "processed_question",
        "answer", "source_pages", "retrieved_chunks",
    }
    required_chunk_fields = {
        "rank", "chunk_id", "score", "pdf_page_start",
        "pdf_page_end", "processed_text",
    }

    if not records:
        raise ValueError(f"Fajl je prazan: {path}")

    for record in records:
        missing = required_record_fields - record.keys()
        if missing:
            raise ValueError(
                f"Pitanje {record.get('question_id')} u {path} nema polja: {sorted(missing)}"
            )

        if len(record["retrieved_chunks"]) < CONTEXT_K:
            raise ValueError(
                f"Pitanje {record['question_id']} ima manje od {CONTEXT_K} chunkova."
            )

        for chunk in record["retrieved_chunks"]:
            missing = required_chunk_fields - chunk.keys()
            if missing:
                raise ValueError(
                    f"Chunk {chunk.get('chunk_id')} u {path} nema polja: {sorted(missing)}"
                )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
retrieval_data: dict[str, dict[str, list[dict]]] = {}

for retriever in RETRIEVERS:
    retrieval_data[retriever] = {}

    for split in SPLITS:
        path = RETRIEVAL_DIR / f"{retriever}_{split}_top{TOP_K_AVAILABLE}.jsonl"

        if not path.exists():
            print(f"Preskočeno: {path} ne postoji.")
            continue

        records = load_jsonl(path)
        validate_retrieval_records(records, path)
        retrieval_data[retriever][split] = records
        print(f"{retriever}/{split}: {len(records)} pitanja")

available_combinations = [
    (retriever, split)
    for retriever, splits in retrieval_data.items()
    for split in splits
]

if not available_combinations:
    raise FileNotFoundError(
        "Nisu pronađeni retrieval rezultati za izabrane splitove."
    )

print("Dostupne kombinacije:", available_combinations)

tfidf/validation: 21 pitanja
bm25/validation: 21 pitanja
dense/validation: 21 pitanja
Dostupne kombinacije: [('tfidf', 'validation'), ('bm25', 'validation'), ('dense', 'validation')]


## Building the RAG Prompt

For each question, the RAG prompt contains the first `CONTEXT_K` ranked chunks and instructs the model to answer in Serbian using only the provided context. The reference answer is excluded from the prompt and is used only for subsequent evaluation.

In [ ]:
SYSTEM_PROMPT = (
    "Ti si asistent koji odgovara na pitanja studenata na osnovu datog "
    "konteksta iz nastavnog materijala. Odgovaraj isključivo na osnovu "
    "konteksta. Ako odgovor nije sadržan u kontekstu, reci da nemaš "
    "dovoljno informacija. Odgovaraj sažeto, jasno i na srpskom jeziku."
)


def build_context(retrieved_chunks: list[dict], context_k: int) -> str:
    if context_k < 1:
        raise ValueError("context_k mora biti pozitivan ceo broj.")

    parts = []
    for chunk in retrieved_chunks[:context_k]:
        parts.append(
            f"[Izvor {chunk['rank']}, strane {chunk['pdf_page_start']}-"
            f"{chunk['pdf_page_end']}]\n{chunk['processed_text'].strip()}"
        )

    return "\n\n".join(parts)


def build_messages(
    question: str,
    retrieved_chunks: list[dict],
    context_k: int,
) -> list[dict]:
    context = build_context(retrieved_chunks, context_k)
    user_prompt = (
        f"Kontekst:\n{context}\n\n"
        f"Pitanje: {question}\n\n"
        "Odgovor:"
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

## Loading Qwen3-8B on the Colab GPU

Qwen3-8B is loaded in 4-bit quantized form to fit within the memory available on the Colab GPU.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available. Select a GPU runtime in Google Colab."
    )

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.eval()
MODEL_INPUT_DEVICE = next(model.parameters()).device

print("Model loaded successfully.")
print("GPU:", torch.cuda.get_device_name(0))
print("Input device:", MODEL_INPUT_DEVICE)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded successfully.
GPU: Tesla T4
Input device: cuda:0


## Checking the Prompt Size

The prompt is tokenized before generation to verify that the selected context fits within the model's context window.

In [15]:
example_retriever, example_split = available_combinations[0]
length_example = retrieval_data[example_retriever][example_split][0]
length_messages = build_messages(
    question=length_example["processed_question"],
    retrieved_chunks=length_example["retrieved_chunks"],
    context_k=CONTEXT_K,
)
prompt_ids = tokenizer.apply_chat_template(
    length_messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(f"Prompt token count: {len(prompt_ids)}")
print(f"Number of context chunks: {CONTEXT_K}")

Prompt token count: 2
Number of context chunks: 5


## Answer Generation Function

The function sends the question and retrieved context to Qwen3-8B on the Colab GPU and returns the generated answer.

In [16]:
@torch.inference_mode()
def generate_answer(
    question: str,
    retrieved_chunks: list[dict],
    context_k: int = CONTEXT_K,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> str:

    messages = build_messages(
        question=question,
        retrieved_chunks=retrieved_chunks,
        context_k=context_k,
    )

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    ).to(MODEL_INPUT_DEVICE)

    pad_token_id = (
        tokenizer.pad_token_id
        if tokenizer.pad_token_id is not None
        else tokenizer.eos_token_id
    )

    output_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=pad_token_id,
    )

    prompt_length = model_inputs["input_ids"].shape[-1]

    generated_ids = output_ids[0, prompt_length:]

    generated_answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return generated_answer.strip()

## Testing on a Validation Example

In [ ]:
if "validation" not in retrieval_data.get("dense", {}):
    raise FileNotFoundError("Dense validation rezultati nisu pronađeni.")

example = retrieval_data["dense"]["validation"][0]
generated = generate_answer(
    question=example["processed_question"],
    retrieved_chunks=example["retrieved_chunks"],
)

print(f"Pitanje: {example['question']}")
print(f"Referentni odgovor: {example['answer']}")
print(f"Generisani odgovor: {generated}")

Pitanje: Šta je Cachegrind, za šta služi i kako se koristi?
Referentni odgovor: Cachegrind je Valgrind alat za profilisanje keš memorije. Koristi se pokretanjem programa kroz Cachegrind, nakon čega se analiziraju prikupljene informacije o pristupima kešu i izvršavanju.
Generisani odgovor: Cachegrind je alat platforme Valgrind koji služi za softversko profajliranje keš memorije. On simulira memorijski podsistem procesora i praćenje pristupa keš memoriji, kao i broj promašaja prilikom čitanja i upisa u različite nivoe keša (npr. L1 i LL). Cachegrind se koristi za identifikaciju uskih grla u performansama vezanim za efikasnost upotrebe keš memorije.


## Answer Generation and Saving

In [22]:
def load_completed_question_ids(path: Path) -> set:
    if not path.exists():
        return set()

    return {record["question_id"] for record in load_jsonl(path)}


def generate_and_save(
    data: list[dict],
    retriever: str,
    split: str,
    context_k: int = CONTEXT_K,
) -> None:
    output_path = GENERATION_DIR / f"{retriever}_{split}_answers.jsonl"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    completed_ids = load_completed_question_ids(output_path)

    remaining = [
        example for example in data
        if example["question_id"] not in completed_ids
    ]
    print(
        f"{retriever}/{split}: završeno {len(completed_ids)}, "
        f"preostalo {len(remaining)}"
    )

    with output_path.open("a", encoding="utf-8") as file:
        for position, example in enumerate(remaining, start=1):
            retrieved_chunks = example["retrieved_chunks"]
            used_chunks = retrieved_chunks[:context_k]
            generated_answer = generate_answer(
                question=example["processed_question"],
                retrieved_chunks=retrieved_chunks,
                context_k=context_k,
            )

            record = {
                "question_id": example["question_id"],
                "question": example["question"],
                "answer": example["answer"],
                "source_pages": example["source_pages"],
                "retriever": retriever,
                "generator": GENERATOR_NAME,
                "generator_model": GENERATOR_MODEL_NAME,
                "split": split,
                "context_k": context_k,
                "context_chunk_ids": [
                    chunk["chunk_id"] for chunk in used_chunks
                ],
                "generated_answer": generated_answer,
            }

            file.write(json.dumps(record, ensure_ascii=False) + "\n")
            file.flush()
            print(
                f"[{position}/{len(remaining)}] "
                f"Sačuvano pitanje {example['question_id']}"
            )

In [ ]:
for retriever, splits in retrieval_data.items():
    for split, data in splits.items():
        generate_and_save(
            data=data,
            retriever=retriever,
            split=split,
        )

tfidf/validation: završeno 0, preostalo 21
[1/21] Sačuvano pitanje 132
[2/21] Sačuvano pitanje 125
[3/21] Sačuvano pitanje 84
[4/21] Sačuvano pitanje 141
[5/21] Sačuvano pitanje 114
[6/21] Sačuvano pitanje 26
[7/21] Sačuvano pitanje 127
[8/21] Sačuvano pitanje 52
[9/21] Sačuvano pitanje 69
[10/21] Sačuvano pitanje 80
[11/21] Sačuvano pitanje 134
[12/21] Sačuvano pitanje 136
[13/21] Sačuvano pitanje 90
[14/21] Sačuvano pitanje 143
[15/21] Sačuvano pitanje 89
[16/21] Sačuvano pitanje 30
[17/21] Sačuvano pitanje 95
[18/21] Sačuvano pitanje 1
[19/21] Sačuvano pitanje 10
[20/21] Sačuvano pitanje 55
[21/21] Sačuvano pitanje 138
bm25/validation: završeno 0, preostalo 21
[1/21] Sačuvano pitanje 132
[2/21] Sačuvano pitanje 125
[3/21] Sačuvano pitanje 84
[4/21] Sačuvano pitanje 141
[5/21] Sačuvano pitanje 114
[6/21] Sačuvano pitanje 26
[7/21] Sačuvano pitanje 127
[8/21] Sačuvano pitanje 52
[9/21] Sačuvano pitanje 69
[10/21] Sačuvano pitanje 80
[11/21] Sačuvano pitanje 134
[12/21] Sačuvano pitanj

## Saving Generation Metadata

In [ ]:
GENERATION_DIR.mkdir(parents=True, exist_ok=True)

metadata = {
    "model_name": GENERATOR_MODEL_NAME,
    "runtime": GENERATOR_RUNTIME,
    "quantized": True,
    "context_k": CONTEXT_K,
    "retrieval_top_k_available": TOP_K_AVAILABLE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "enable_thinking": False,
    "run_test": RUN_TEST,
    "quantization_bits": 4,
    "retrievers_processed": [
        retriever
        for retriever, splits in retrieval_data.items()
        if splits
    ],
    "splits_processed": list(SPLITS),
}

with (GENERATION_DIR / "qwen_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Sačuvani metapodaci generisanja.")

Sačuvani metapodaci generisanja.


## Test Generation with the Original Prompt

The original prompt configuration is now applied to the held-out test set. Results are saved separately for TF-IDF, BM25, and dense retrieval so that they can later be compared with the concise-prompt variant.


In [23]:
TEST_SPLIT = "test"
PROMPT_VARIANT = "original"

test_retrieval_data = {}

for retriever in RETRIEVERS:
    path = (
        RETRIEVAL_DIR
        / f"{retriever}_{TEST_SPLIT}_top{TOP_K_AVAILABLE}.jsonl"
    )

    if not path.exists():
        print(f"Preskočeno: {path} ne postoji.")
        continue

    records = load_jsonl(path)

    validate_retrieval_records(
        records=records,
        path=path,
    )

    test_retrieval_data[retriever] = records

    print(
        f"{retriever}/{TEST_SPLIT}: "
        f"{len(records)} pitanja"
    )

if not test_retrieval_data:
    raise FileNotFoundError(
        "Nisu pronađeni retrieval rezultati za test skup."
    )

tfidf/test: 22 pitanja
bm25/test: 22 pitanja
dense/test: 22 pitanja


In [24]:
def generate_test_variant_and_save(
    data: list[dict],
    retriever: str,
    prompt_variant: str,
    context_k: int = CONTEXT_K,
) -> None:

    output_path = (
        GENERATION_DIR
        / (
            f"{retriever}_test_"
            f"{prompt_variant}_answers.jsonl"
        )
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    completed_ids = load_completed_question_ids(
        output_path
    )

    remaining = [
        example
        for example in data
        if example["question_id"] not in completed_ids
    ]

    print(
        f"{retriever}/test/{prompt_variant}: "
        f"završeno {len(completed_ids)}, "
        f"preostalo {len(remaining)}"
    )

    with output_path.open(
        "a",
        encoding="utf-8",
    ) as file:

        for position, example in enumerate(
            remaining,
            start=1,
        ):
            retrieved_chunks = example[
                "retrieved_chunks"
            ]

            used_chunks = retrieved_chunks[
                :context_k
            ]

            generated_answer = generate_answer(
                question=example[
                    "processed_question"
                ],
                retrieved_chunks=retrieved_chunks,
                context_k=context_k,
            )

            record = {
                "question_id": example["question_id"],
                "question": example["question"],
                "answer": example["answer"],
                "source_pages": example["source_pages"],
                "retriever": retriever,
                "generator": GENERATOR_NAME,
                "generator_model": GENERATOR_MODEL_NAME,
                "split": "test",
                "prompt_variant": prompt_variant,
                "context_k": context_k,
                "context_chunk_ids": [
                    chunk["chunk_id"]
                    for chunk in used_chunks
                ],
                "generated_answer": generated_answer,
            }

            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

            file.flush()

            print(
                f"[{position}/{len(remaining)}] "
                f"Sačuvano pitanje "
                f"{example['question_id']}"
            )

In [25]:
for retriever, data in test_retrieval_data.items():
    generate_test_variant_and_save(
        data=data,
        retriever=retriever,
        prompt_variant=PROMPT_VARIANT,
    )

tfidf/test/original: završeno 0, preostalo 22
[1/22] Sačuvano pitanje 61
[2/22] Sačuvano pitanje 25
[3/22] Sačuvano pitanje 27
[4/22] Sačuvano pitanje 130
[5/22] Sačuvano pitanje 33
[6/22] Sačuvano pitanje 5
[7/22] Sačuvano pitanje 124
[8/22] Sačuvano pitanje 83
[9/22] Sačuvano pitanje 91
[10/22] Sačuvano pitanje 73
[11/22] Sačuvano pitanje 92
[12/22] Sačuvano pitanje 34
[13/22] Sačuvano pitanje 106
[14/22] Sačuvano pitanje 9
[15/22] Sačuvano pitanje 64
[16/22] Sačuvano pitanje 7
[17/22] Sačuvano pitanje 46
[18/22] Sačuvano pitanje 119
[19/22] Sačuvano pitanje 53
[20/22] Sačuvano pitanje 103
[21/22] Sačuvano pitanje 107
[22/22] Sačuvano pitanje 118
bm25/test/original: završeno 0, preostalo 22
[1/22] Sačuvano pitanje 61
[2/22] Sačuvano pitanje 25
[3/22] Sačuvano pitanje 27
[4/22] Sačuvano pitanje 130
[5/22] Sačuvano pitanje 33
[6/22] Sačuvano pitanje 5
[7/22] Sačuvano pitanje 124
[8/22] Sačuvano pitanje 83
[9/22] Sačuvano pitanje 91
[10/22] Sačuvano pitanje 73
[11/22] Sačuvano pitanje 9

In [26]:
test_metadata = {
    "model_name": GENERATOR_MODEL_NAME,
    "runtime": GENERATOR_RUNTIME,
    "quantized": True,
    "quantization_bits": 4,
    "split": TEST_SPLIT,
    "prompt_variant": PROMPT_VARIANT,
    "system_prompt": SYSTEM_PROMPT,
    "context_k": CONTEXT_K,
    "retrieval_top_k_available": TOP_K_AVAILABLE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "enable_thinking": False,
    "retrievers_processed": list(
        test_retrieval_data.keys()
    ),
}

metadata_path = (
    GENERATION_DIR
    / f"qwen_test_{PROMPT_VARIANT}_metadata.json"
)

with metadata_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        test_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    f"Sačuvani metapodaci: {metadata_path}"
)

Sačuvani metapodaci: /content/drive/MyDrive/Student-Question-Answering-from-Course-Materials/data/generation/qwen_colab/qwen_test_original_metadata.json


## Test Generation with the Concise Prompt

This variant uses the same model, retrieval results, `CONTEXT_K`, and generation
parameters as the original variant. Only the system prompt is changed to
require shorter and more direct answers.

In [27]:
PROMPT_VARIANT = "concise"

SYSTEM_PROMPT = (
    "Ti si asistent koji odgovara na pitanja studenata na osnovu datog "
    "konteksta iz nastavnog materijala. Odgovaraj isključivo na osnovu "
    "konteksta. Ako odgovor nije sadržan u kontekstu, reci da nemaš "
    "dovoljno informacija. Odgovaraj sažeto — jednom do dve rečenice, "
    "bez uvodnih fraza i bez dodatnih primera osim ako pitanje to "
    "eksplicitno traži. Odgovaraj jasno i na srpskom jeziku."
)

print(
    f"Aktivna prompt varijanta: {PROMPT_VARIANT}"
)

Aktivna prompt varijanta: concise


In [28]:
for retriever, data in test_retrieval_data.items():
    generate_test_variant_and_save(
        data=data,
        retriever=retriever,
        prompt_variant=PROMPT_VARIANT,
    )

tfidf/test/concise: završeno 0, preostalo 22
[1/22] Sačuvano pitanje 61
[2/22] Sačuvano pitanje 25
[3/22] Sačuvano pitanje 27
[4/22] Sačuvano pitanje 130
[5/22] Sačuvano pitanje 33
[6/22] Sačuvano pitanje 5
[7/22] Sačuvano pitanje 124
[8/22] Sačuvano pitanje 83
[9/22] Sačuvano pitanje 91
[10/22] Sačuvano pitanje 73
[11/22] Sačuvano pitanje 92
[12/22] Sačuvano pitanje 34
[13/22] Sačuvano pitanje 106
[14/22] Sačuvano pitanje 9
[15/22] Sačuvano pitanje 64
[16/22] Sačuvano pitanje 7
[17/22] Sačuvano pitanje 46
[18/22] Sačuvano pitanje 119
[19/22] Sačuvano pitanje 53
[20/22] Sačuvano pitanje 103
[21/22] Sačuvano pitanje 107
[22/22] Sačuvano pitanje 118
bm25/test/concise: završeno 0, preostalo 22
[1/22] Sačuvano pitanje 61
[2/22] Sačuvano pitanje 25
[3/22] Sačuvano pitanje 27
[4/22] Sačuvano pitanje 130
[5/22] Sačuvano pitanje 33
[6/22] Sačuvano pitanje 5
[7/22] Sačuvano pitanje 124
[8/22] Sačuvano pitanje 83
[9/22] Sačuvano pitanje 91
[10/22] Sačuvano pitanje 73
[11/22] Sačuvano pitanje 92


In [29]:

concise_test_metadata = {
    "model_name": GENERATOR_MODEL_NAME,
    "runtime": GENERATOR_RUNTIME,
    "quantized": True,
    "quantization_bits": 4,
    "split": TEST_SPLIT,
    "prompt_variant": PROMPT_VARIANT,
    "system_prompt": SYSTEM_PROMPT,
    "context_k": CONTEXT_K,
    "retrieval_top_k_available": TOP_K_AVAILABLE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "enable_thinking": False,
    "retrievers_processed": list(
        test_retrieval_data.keys()
    ),
}

concise_metadata_path = (
    GENERATION_DIR
    / f"qwen_test_{PROMPT_VARIANT}_metadata.json"
)

with concise_metadata_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        concise_test_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    f"Sačuvani metapodaci: "
    f"{concise_metadata_path}"
)

Sačuvani metapodaci: /content/drive/MyDrive/Student-Question-Answering-from-Course-Materials/data/generation/qwen_colab/qwen_test_concise_metadata.json
